# Supervisor Demo Notebook: Model 1 -> Model 5

This notebook is a complete runnable demonstration with:
1. Model 1: line/cartouche detection + prediction vs GT visualization
2. Model 2: glyph detection + enhanced crop generation
3. Model 3: glyph classification (single and folder)
4. Model 4: Gardiner sequence -> transliteration -> English translation
5. Model 5: grammar correction

Run cells in order from top to bottom.

In [ ]:
# Install dependencies (Kaggle-compatible)
!pip -q install ultralytics roboflow transformers sentencepiece accelerate requests pandas

## Configuration
Set your API keys and verify all paths.

- `ROBOFLOW_API_KEY`: required for downloading Model 1 dataset and GT labels
- `HF_TOKEN` is optional unless your environment requires authenticated model pulls

In [ ]:
import os

# API keys
ROBOFLOW_API_KEY = "YOUR_ROBOFLOW_API_KEY"
HF_TOKEN = ""

# Model weight paths
MODEL1_LINE_CARTOUCHE_WEIGHTS = "/kaggle/input/notebooks/aliasifkhan131/egyptiananalysispolygon/runs/segment/train/weights/best.pt"
MODEL2_GLYPH_DETECT_WEIGHTS = "/kaggle/input/notebooks/aliahnafkhan/egyptiananalysis/runs/detect/train/weights/best.pt"
MODEL3_CLASSIFIER_CKPT = "/kaggle/input/notebooks/aliasifkhan131/resnet50-train/checkpoints/resnet50/best_model.pth"
MODEL3_CLASS_DIR = "/kaggle/input/notebooks/aliasifkhan131/resnet50-train/Glyph2025"

# Translation/grammar models
TRANSLATION_MODEL_NAME = "Asiif/mt5_hieroglyph"
TRANSLATION_MODEL_COMMIT = "846f2dd0bc50c77fff26d1188021232955e41e19"
GRAMMAR_MODEL_NAME = "google/flan-t5-base"

# Inference settings
TASK_PREFIX = "translate Egyptian transliteration to English: "
MAX_SOURCE_LEN = 128
MAX_TARGET_LEN = 64

# Working directories
WORK_ROOT = "/kaggle/working"
MODEL1_CROP_OUTPUT = os.path.join(WORK_ROOT, "model1_line_crops")
MODEL2_GLYPH_CROP_OUTPUT = os.path.join(WORK_ROOT, "extracted_images")
os.makedirs(MODEL1_CROP_OUTPUT, exist_ok=True)
os.makedirs(MODEL2_GLYPH_CROP_OUTPUT, exist_ok=True)

# Demo image paths (set these if different in your environment)
DEMO_WALL_IMAGE_PATH = "/kaggle/working/sample_hieroglyphs.jpg"
DEMO_LINE_IMAGE_PATH = "/kaggle/input/datasets/aliasifkhan2105131/line-cartoush-ds/crops/Line/0ebc2341775c4c9ab0c8fc788cba5ca32.jpg"

print("Configuration loaded.")

In [ ]:
import cv2
import math
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from pathlib import Path
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional

import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms

from PIL import Image
from ultralytics import YOLO
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [ ]:
def download_demo_wall_image(out_path: str = DEMO_WALL_IMAGE_PATH):
    url = "https://raw.githubusercontent.com/JamesPiggott/Ancient-Language-Decipherer/master/examples/sample_hieroglyphs.jpg"
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    with open(out_path, "wb") as f:
        f.write(r.content)
    print(f"Downloaded demo wall image: {out_path}")

download_demo_wall_image()

In [ ]:
# ------------------------------
# Shared utility helpers
# ------------------------------
def clear_folder(folder: str):
    os.makedirs(folder, exist_ok=True)
    for name in os.listdir(folder):
        p = os.path.join(folder, name)
        if os.path.isfile(p):
            os.remove(p)

def plot_boxes(ax, image_rgb, boxes, color='red', linewidth=2, title=''):
    ax.imshow(image_rgb)
    for b in boxes:
        x1, y1, x2, y2 = b
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=linewidth, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
    ax.set_title(title)
    ax.axis('off')

def yolo_xywhn_to_xyxy(xc, yc, w, h, img_w, img_h):
    bw = w * img_w
    bh = h * img_h
    x1 = (xc * img_w) - bw / 2
    y1 = (yc * img_h) - bh / 2
    x2 = x1 + bw
    y2 = y1 + bh
    return [x1, y1, x2, y2]

def polygon_norm_to_xyxy(coords, img_w, img_h):
    xs = [coords[i] * img_w for i in range(0, len(coords), 2)]
    ys = [coords[i] * img_h for i in range(1, len(coords), 2)]
    return [min(xs), min(ys), max(xs), max(ys)]

def read_gt_boxes(label_path: str, img_w: int, img_h: int):
    boxes = []
    if not os.path.exists(label_path):
        return boxes
    with open(label_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            vals = [float(x) for x in parts[1:]]
            if len(vals) == 4:
                boxes.append(yolo_xywhn_to_xyxy(vals[0], vals[1], vals[2], vals[3], img_w, img_h))
            elif len(vals) >= 6 and len(vals) % 2 == 0:
                boxes.append(polygon_norm_to_xyxy(vals, img_w, img_h))
    return boxes

In [ ]:
# ------------------------------
# Model loading (2, 3, 4, 5)
# ------------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

model2 = YOLO(MODEL2_GLYPH_DETECT_WEIGHTS)

if os.path.exists(MODEL3_CLASS_DIR):
    class_names = sorted([d for d in os.listdir(MODEL3_CLASS_DIR) if os.path.isdir(os.path.join(MODEL3_CLASS_DIR, d))])
else:
    class_names = None
print('Class names loaded:', 0 if class_names is None else len(class_names))

classification_model = models.resnet50(weights=None)
classification_model.fc = nn.Sequential(
    nn.Linear(2048, 512),
    nn.ReLU(inplace=True),
    nn.Dropout(0.4),
    nn.Linear(512, 353),
)
ckpt = torch.load(MODEL3_CLASSIFIER_CKPT, map_location=device)
classification_model.load_state_dict(ckpt['model_state_dict'])
classification_model = classification_model.to(device).eval()

classify_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

tok_kwargs = {'use_fast': False, 'revision': TRANSLATION_MODEL_COMMIT}
mdl_kwargs = {'revision': TRANSLATION_MODEL_COMMIT}
if HF_TOKEN:
    tok_kwargs['token'] = HF_TOKEN
    mdl_kwargs['token'] = HF_TOKEN

tokenizer = AutoTokenizer.from_pretrained(TRANSLATION_MODEL_NAME, **tok_kwargs)
tr_model = AutoModelForSeq2SeqLM.from_pretrained(TRANSLATION_MODEL_NAME, **mdl_kwargs).to(device)

g_tok_kwargs = {}
g_mdl_kwargs = {}
if HF_TOKEN:
    g_tok_kwargs['token'] = HF_TOKEN
    g_mdl_kwargs['token'] = HF_TOKEN
flan_tokenizer = AutoTokenizer.from_pretrained(GRAMMAR_MODEL_NAME, **g_tok_kwargs)
flan_model = AutoModelForSeq2SeqLM.from_pretrained(GRAMMAR_MODEL_NAME, **g_mdl_kwargs).to(device)

print('Model 2, 3, 4, 5 loaded successfully.')

In [ ]:
# ------------------------------
# Model 1: line/cartouche demo with GT
# ------------------------------
def download_roboflow_dataset(api_key: str, version: int = 1, fmt: str = 'yolov11'):
    if not api_key or api_key == 'YOUR_ROBOFLOW_API_KEY':
        raise ValueError('Set ROBOFLOW_API_KEY before running this cell.')
    from roboflow import Roboflow
    rf = Roboflow(api_key=api_key)
    project = rf.workspace('pyramid-cpfw6').project('pyramidpolygon')
    ds = project.version(version).download(fmt)
    return ds.location

def load_model1(weights_path: str = MODEL1_LINE_CARTOUCHE_WEIGHTS):
    return YOLO(weights_path)

def demo_model1_prediction_vs_gt(image_path: str, dataset_root: str, model1=None, conf: float = 0.25):
    model1 = model1 or load_model1()

    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        raise FileNotFoundError(f'Image not found: {image_path}')
    h, w = img_bgr.shape[:2]
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    pred = model1.predict(source=image_path, conf=conf, verbose=False)[0]
    pred_boxes = pred.boxes.xyxy.cpu().numpy().tolist() if pred.boxes is not None else []

    stem = Path(image_path).stem
    gt_label = os.path.join(dataset_root, 'valid', 'labels', f'{stem}.txt')
    if not os.path.exists(gt_label):
        gt_label = os.path.join(dataset_root, 'train', 'labels', f'{stem}.txt')
    gt_boxes = read_gt_boxes(gt_label, w, h)

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    plot_boxes(axes[0], img_rgb, pred_boxes, color='red', title='Model 1 Prediction (Red)')
    plot_boxes(axes[1], img_rgb, gt_boxes, color='green', title='Ground Truth (Green)')
    plt.tight_layout()
    plt.show()

    print('Pred boxes:', len(pred_boxes), '| GT boxes:', len(gt_boxes))
    return {'pred_boxes': pred_boxes, 'gt_boxes': gt_boxes, 'gt_label_path': gt_label}

In [ ]:
# Model 1 demo execution
# 1) Download dataset with labels
# 2) Choose one valid image from dataset
# 3) Visualize prediction (red) vs ground truth (green)

roboflow_root = download_roboflow_dataset(ROBOFLOW_API_KEY, version=1, fmt='yolov11')
model1 = load_model1()

valid_img_dir = os.path.join(roboflow_root, 'valid', 'images')
valid_imgs = sorted([os.path.join(valid_img_dir, x) for x in os.listdir(valid_img_dir) if x.lower().endswith(('.jpg', '.jpeg', '.png'))])
if not valid_imgs:
    raise RuntimeError('No validation images found in downloaded Roboflow dataset.')

model1_demo_image = valid_imgs[0]
print('Using Model 1 demo image:', model1_demo_image)
model1_result = demo_model1_prediction_vs_gt(model1_demo_image, roboflow_root, model1=model1, conf=0.25)

In [ ]:
# ------------------------------
# Model 2: glyph detection and enhanced crops
# ------------------------------
def save_line_crops_from_wall_image(wall_image_path: str, model1, out_dir: str = MODEL1_CROP_OUTPUT, conf: float = 0.25):
    clear_folder(out_dir)

    img = cv2.imread(wall_image_path)
    if img is None:
        raise FileNotFoundError(f'Wall image not found: {wall_image_path}')

    pred = model1.predict(source=wall_image_path, conf=conf, verbose=False)[0]
    boxes = pred.boxes.xyxy.cpu().numpy().astype(int) if pred.boxes is not None else np.empty((0, 4), dtype=int)

    saved = []
    for i, (x1, y1, x2, y2) in enumerate(boxes):
        crop = img[max(0, y1):max(0, y2), max(0, x1):max(0, x2)]
        if crop.size == 0:
            continue
        out = os.path.join(out_dir, f'line_crop_{i:03d}.png')
        cv2.imwrite(out, crop)
        saved.append(out)

    print('Saved line crops from Model 1:', len(saved))
    return saved

def detect_glyphs_and_save_enhanced(line_image_path: str, model2, out_dir: str = MODEL2_GLYPH_CROP_OUTPUT, conf: float = 0.4, margin: int = 4, scale_factor: int = 3, visualize: bool = True):
    clear_folder(out_dir)

    det = model2.predict(source=line_image_path, conf=conf, verbose=False)[0]
    original = cv2.imread(line_image_path)
    if original is None:
        raise FileNotFoundError(f'Line image not found: {line_image_path}')

    img_h, img_w = original.shape[:2]
    boxes = det.boxes.xyxy.cpu().numpy().astype(int) if det.boxes is not None else np.empty((0, 4), dtype=int)

    drawn = original.copy()
    saved = []
    for i, (x1, y1, x2, y2) in enumerate(boxes):
        x1 = max(0, x1 - margin)
        y1 = max(0, y1 - margin)
        x2 = min(img_w, x2 + margin)
        y2 = min(img_h, y2 + margin)

        crop = original[y1:y2, x1:x2]
        if crop.shape[0] < 5 or crop.shape[1] < 5:
            continue

        new_w = crop.shape[1] * scale_factor
        new_h = crop.shape[0] * scale_factor
        crop = cv2.resize(crop, (new_w, new_h), interpolation=cv2.INTER_LANCZOS4)

        gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
        enhanced = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)

        out = os.path.join(out_dir, f'ROI_{i}.png')
        cv2.imwrite(out, enhanced)
        saved.append(out)

        cv2.rectangle(drawn, (x1, y1), (x2, y2), (0, 0, 255), 2)

    if visualize:
        plt.figure(figsize=(10, 6))
        plt.imshow(cv2.cvtColor(drawn, cv2.COLOR_BGR2RGB))
        plt.title('Model 2 glyph boxes (red)')
        plt.axis('off')
        plt.show()

    print('Enhanced glyph crops saved:', len(saved))
    return {'boxes': boxes.tolist(), 'enhanced_paths': saved}

In [ ]:
# Model 2 demo execution
# Option A: use a known line image path
line_image_for_model2 = DEMO_LINE_IMAGE_PATH
if not os.path.exists(line_image_for_model2):
    # Option B: generate line crops from wall image via Model 1 and take first crop
    generated_line_crops = save_line_crops_from_wall_image(DEMO_WALL_IMAGE_PATH, model1=model1, out_dir=MODEL1_CROP_OUTPUT, conf=0.25)
    if not generated_line_crops:
        raise RuntimeError('No line crops generated by Model 1. Please provide a valid line image path manually.')
    line_image_for_model2 = generated_line_crops[0]

print('Using line image for Model 2:', line_image_for_model2)
model2_result = detect_glyphs_and_save_enhanced(line_image_for_model2, model2=model2, out_dir=MODEL2_GLYPH_CROP_OUTPUT, conf=0.4, margin=4, scale_factor=3, visualize=True)

In [ ]:
# ------------------------------
# Model 3: classification
# ------------------------------
def predict_resnet50_single(image_path: str):
    input_image = Image.open(image_path).convert('RGB')
    input_tensor = classify_transform(input_image).unsqueeze(0).to(device)
    with torch.no_grad():
        outputs = classification_model(input_tensor)
    _, pred_idx = torch.max(outputs, 1)
    return int(pred_idx.item())

def classify_single_path(image_path: str):
    idx = predict_resnet50_single(image_path)
    cls = class_names[idx] if class_names is not None and idx < len(class_names) else str(idx)
    print('Single image:', image_path)
    print('Predicted index:', idx)
    print('Predicted class:', cls)
    return {'path': image_path, 'index': idx, 'class_name': cls}

def classify_folder(folder_path: str):
    files = sorted([os.path.join(folder_path, x) for x in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, x))])
    results = []
    for f in files:
        idx = predict_resnet50_single(f)
        cls = class_names[idx] if class_names is not None and idx < len(class_names) else str(idx)
        results.append({'path': f, 'index': idx, 'class_name': cls})
    print('Total classified crops:', len(results))
    print('Class sequence:', [r['class_name'] for r in results])
    return results

def show_classified_gray_crops(classification_results: List[Dict], max_items: int = 24, cols: int = 4):
    if not classification_results:
        print('No classification results to display.')
        return

    items = classification_results[:max_items]
    n = len(items)
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3.2 * rows))
    if rows == 1 and cols == 1:
        axes = np.array([[axes]])
    elif rows == 1:
        axes = np.array([axes])
    elif cols == 1:
        axes = np.array([[ax] for ax in axes])

    for i in range(rows * cols):
        r = i // cols
        c = i % cols
        ax = axes[r][c]
        ax.axis('off')
        if i >= n:
            continue

        item = items[i]
        img = cv2.imread(item['path'], cv2.IMREAD_GRAYSCALE)
        if img is None:
            ax.set_title('Unreadable image', fontsize=10)
            continue

        ax.imshow(img, cmap='gray')
        ax.set_title(f"{Path(item['path']).name}\nPred: {item['class_name']} (idx={item['index']})", fontsize=9)

    plt.tight_layout()
    plt.show()

In [ ]:
# Model 3 demo execution with annotated gray crops
if not model2_result['enhanced_paths']:
    raise RuntimeError('No enhanced crops found. Run Model 2 demo first.')

single_class_demo = classify_single_path(model2_result['enhanced_paths'][0])
folder_class_demo = classify_folder(MODEL2_GLYPH_CROP_OUTPUT)
show_classified_gray_crops(folder_class_demo, max_items=24, cols=4)

In [ ]:
# ------------------------------
# Model 4 preparation: dictionary-based Gardiner -> transliteration resolver
# ------------------------------
RAW_BASE = 'https://raw.githubusercontent.com/SeveredAsif/HieroGlyphTranslator/main'
FILES = {
    'train_egy': 'aligned_train_egy.csv',
    'train_eng': 'aligned_train_eng.csv',
    'val_egy': 'aligned_val_egy.csv',
    'val_eng': 'aligned_val_eng.csv',
    'test_egy': 'aligned_test_egy.csv',
    'test_eng': 'aligned_test_eng.csv',
    'dict': 'ancient_egypt_dictionary_compiled.csv',
}

def ensure_local_data():
    candidates = [
        '/kaggle/input/hieroglyphtranslator',
        '/kaggle/input/hieroglyph-translator',
        '/kaggle/input/hieroglyphtranslatordataset',
    ]
    for c in candidates:
        if os.path.isdir(c):
            return c

    work_dir = '/kaggle/working/hieroglyph_data'
    os.makedirs(work_dir, exist_ok=True)
    for _, fname in FILES.items():
        out_path = os.path.join(work_dir, fname)
        if os.path.exists(out_path):
            continue
        url = f'{RAW_BASE}/{fname}'
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        with open(out_path, 'wb') as f:
            f.write(r.content)
    return work_dir

def read_lines(path: str):
    with open(path, 'r', encoding='utf-8') as f:
        return [line.strip() for line in f if line.strip()]

DATA_DIR = ensure_local_data()
train_egy = read_lines(os.path.join(DATA_DIR, FILES['train_egy']))
train_eng = read_lines(os.path.join(DATA_DIR, FILES['train_eng']))
val_egy = read_lines(os.path.join(DATA_DIR, FILES['val_egy']))
val_eng = read_lines(os.path.join(DATA_DIR, FILES['val_eng']))
test_egy = read_lines(os.path.join(DATA_DIR, FILES['test_egy']))
test_eng = read_lines(os.path.join(DATA_DIR, FILES['test_eng']))

assert len(train_egy) == len(train_eng), 'Train split misaligned'
assert len(val_egy) == len(val_eng), 'Val split misaligned'
assert len(test_egy) == len(test_eng), 'Test split misaligned'

SPLIT_DATA = {
    'train': {'egy': train_egy, 'eng': train_eng},
    'val': {'egy': val_egy, 'eng': val_eng},
    'test': {'egy': test_egy, 'eng': test_eng},
}

import re
sign_pattern = re.compile(r'[A-Za-z]{1,2}\d+[A-Za-z]?', flags=re.IGNORECASE)

def canonicalize_sign(token: str) -> str:
    t = str(token).strip()
    m = re.fullmatch(r'([A-Za-z]{1,2})(\d+)([A-Za-z]?)', t)
    if not m:
        return t
    fam, num, suf = m.groups()
    fam = fam[0].upper() + (fam[1:].lower() if len(fam) > 1 else '')
    suf = suf.upper() if suf else ''
    return f'{fam}{num}{suf}'

def tokenize_signs(text: str):
    raw = sign_pattern.findall(str(text))
    if raw:
        return [canonicalize_sign(x) for x in raw]
    compact = re.sub(r'[^A-Za-z0-9]', '', str(text))
    chunks = sign_pattern.findall(compact)
    if chunks:
        return [canonicalize_sign(x) for x in chunks]
    return []

def expand_gardiner_variants(raw_key: str):
    parts = re.split(r'\s+or\s+|/|\|', str(raw_key).strip(), flags=re.IGNORECASE)
    variants = []
    for p in parts:
        toks = tokenize_signs(p.strip())
        if toks:
            variants.append(tuple(toks))
    return variants

df_dict = pd.read_csv(os.path.join(DATA_DIR, FILES['dict']))
df_dict = df_dict.dropna(subset=['Gardiner', 'Transliteration'])

phrase_to_trans = defaultdict(Counter)
for _, row in df_dict.iterrows():
    gard = str(row['Gardiner']).strip()
    trans = str(row['Transliteration']).strip()
    if not gard or not trans:
        continue
    for variant in expand_gardiner_variants(gard):
        phrase_to_trans[variant][trans] += 1

max_phrase_len = max(len(k) for k in phrase_to_trans.keys())

unigram = Counter()
bigram = Counter()
for line in train_egy + val_egy:
    toks = ['<s>'] + [t for t in line.split() if t] + ['</s>']
    for i in range(len(toks) - 1):
        unigram[toks[i]] += 1
        bigram[(toks[i], toks[i + 1])] += 1
    unigram[toks[-1]] += 1
vocab_size = len(unigram)

known_signs = sorted({s for phrase in phrase_to_trans.keys() for s in phrase})
known_sign_set = set(known_signs)
known_families = defaultdict(list)
for s in known_signs:
    m = re.fullmatch(r'([A-Za-z]{1,2})(\d+)([A-Za-z]?)', s)
    if m:
        fam, num, suf = m.groups()
        known_families[(fam, suf)].append((int(num), s))
for k in known_families:
    known_families[k] = sorted(known_families[k], key=lambda x: x[0])

phrase_keys_by_len = defaultdict(list)
for key in phrase_to_trans.keys():
    phrase_keys_by_len[len(key)].append(key)

print('Dictionary loaded. Unique phrase keys:', len(phrase_to_trans))
print('Split sizes:', {k: len(v['egy']) for k, v in SPLIT_DATA.items()})

In [ ]:
# ------------------------------
# Model 4: resolve signs + reorder + translate
# ------------------------------
def split_translit_tokens(line: str):
    return [t for t in line.strip().split() if t]

def log_bigram_prob(prev_tok: str, tok: str, alpha: float = 0.1):
    num = bigram[(prev_tok, tok)] + alpha
    den = unigram[prev_tok] + alpha * max(1, vocab_size)
    return math.log(num / den)

def score_translit_sequence(prev_tok: str, tokens: List[str]):
    s = 0.0
    p = prev_tok
    for t in tokens:
        s += log_bigram_prob(p, t)
        p = t
    return s

def closest_known_sign(raw_sign: str, max_num_gap: int = 3):
    s = canonicalize_sign(raw_sign)
    if s in known_sign_set:
        return s
    m = re.fullmatch(r'([A-Za-z]{1,2})(\d+)([A-Za-z]?)', s)
    if not m:
        return s
    fam, num, suf = m.groups()
    num = int(num)
    bucket = known_families.get((fam, suf), [])
    if not bucket:
        return s
    best_num, best_sign = min(bucket, key=lambda x: abs(x[0] - num))
    return best_sign if abs(best_num - num) <= max_num_gap else s

def normalize_input_signs(signs: List[str]):
    fixed, changes = [], []
    for s in signs:
        c = closest_known_sign(s)
        fixed.append(c)
        if canonicalize_sign(s) != c:
            changes.append((s, c))
    return fixed, changes

def sign_distance(a: str, b: str):
    if a == b:
        return 0.0
    pa = re.fullmatch(r'([A-Za-z]{1,2})(\d+)([A-Za-z]?)', a)
    pb = re.fullmatch(r'([A-Za-z]{1,2})(\d+)([A-Za-z]?)', b)
    if not pa or not pb:
        return 3.0
    fam_a, num_a, suf_a = pa.groups()
    fam_b, num_b, suf_b = pb.groups()
    dist = 0.0
    if fam_a != fam_b:
        dist += 2.0
    if suf_a != suf_b:
        dist += 0.5
    dist += min(abs(int(num_a) - int(num_b)) / 6.0, 2.0)
    return dist

def get_fuzzy_phrase_candidates(normalized: List[str], pos: int, max_candidates: int = 16):
    n = len(normalized)
    cands = []
    max_l = min(max_phrase_len, n - pos)
    for l in range(max_l, 0, -1):
        segment = normalized[pos:pos + l]
        for key in phrase_keys_by_len.get(l, []):
            d = sum(sign_distance(segment[i], key[i]) for i in range(l)) / l
            cands.append((key, l, d))
    cands.sort(key=lambda x: (x[2], -x[1]))
    return cands[:max_candidates]

@dataclass
class Hyp:
    pos: int
    prev: str
    out: List[str]
    score: float
    segments: List[Dict]

def resolve_sign_sequence(signs: List[str], beam_size: int = 6, span_bonus: float = 2.5, lm_weight: float = 1.0, fuzzy_penalty: float = 2.5):
    if not signs:
        return [], 0.0, [], []

    normalized, changes = normalize_input_signs(signs)
    beams = [Hyp(pos=0, prev='<s>', out=[], score=0.0, segments=[])]
    n = len(normalized)

    while True:
        if all(h.pos >= n for h in beams):
            break

        new_beams = []
        for h in beams:
            if h.pos >= n:
                new_beams.append(h)
                continue

            found_any = False
            max_l = min(max_phrase_len, n - h.pos)
            for l in range(max_l, 0, -1):
                key = tuple(normalized[h.pos:h.pos + l])
                if key not in phrase_to_trans:
                    continue
                found_any = True
                for tr, freq in phrase_to_trans[key].items():
                    tr_toks = split_translit_tokens(tr)
                    if not tr_toks:
                        continue
                    lm = score_translit_sequence(h.prev, tr_toks)
                    sc = h.score + (lm_weight * lm) + 0.2 * math.log(freq + 1.0) + span_bonus * l
                    seg = {'start': h.pos, 'end': h.pos + l, 'input_signs': list(normalized[h.pos:h.pos+l]), 'matched_key': list(key), 'translit': tr, 'exact': True, 'distance': 0.0}
                    new_beams.append(Hyp(pos=h.pos + l, prev=tr_toks[-1], out=h.out + tr_toks, score=sc, segments=h.segments + [seg]))

            if not found_any:
                fuzzy = get_fuzzy_phrase_candidates(normalized, h.pos, max_candidates=16)
                for key, l, dist in fuzzy:
                    for tr, freq in phrase_to_trans[key].items():
                        tr_toks = split_translit_tokens(tr)
                        if not tr_toks:
                            continue
                        lm = score_translit_sequence(h.prev, tr_toks)
                        sc = h.score + (lm_weight * lm) + 0.2 * math.log(freq + 1.0) + span_bonus * l - fuzzy_penalty * dist
                        seg = {'start': h.pos, 'end': h.pos + l, 'input_signs': list(normalized[h.pos:h.pos+l]), 'matched_key': list(key), 'translit': tr, 'exact': False, 'distance': float(dist)}
                        new_beams.append(Hyp(pos=h.pos + l, prev=tr_toks[-1], out=h.out + tr_toks, score=sc, segments=h.segments + [seg]))

        if not new_beams:
            new_beams = [Hyp(pos=n, prev='</s>', out=[], score=-1e9, segments=[])]

        new_beams = sorted(new_beams, key=lambda x: x.score, reverse=True)[:beam_size]
        beams = new_beams

    best = max(beams, key=lambda x: x.score)
    return best.out, best.score, changes, best.segments

def parse_gardiner_input(gardiner_input):
    if isinstance(gardiner_input, list):
        return [canonicalize_sign(str(x).strip()) for x in gardiner_input if str(x).strip()]
    text = str(gardiner_input).strip()
    if not text:
        return []
    parts = [p for p in text.split() if p]
    if parts:
        return [canonicalize_sign(p) for p in parts]
    return tokenize_signs(text)

def _decode_signs(signs: List[str], decode_beam_size: int = 6):
    out_tokens, score, changes, segments = resolve_sign_sequence(signs, beam_size=decode_beam_size)
    translit = ' '.join(out_tokens).strip()
    exact_ratio = (sum(1 for s in segments if s['exact']) / max(1, len(segments)))
    return translit, score, changes, segments, exact_ratio

def _generate_local_orders(n: int, max_combinations: int = 20, n_segments: int = 3, max_shuffle_span: int = 3):
    base = tuple(range(n))
    if n <= 1 or max_combinations <= 1:
        return [base]

    rng = np.random.default_rng(42 + n)
    orders = [base]
    seen = {base}
    segments = [arr.tolist() for arr in np.array_split(np.arange(n), max(1, min(n_segments, n)))]

    attempts = 0
    max_attempts = max_combinations * 25
    while len(orders) < max_combinations and attempts < max_attempts:
        attempts += 1
        order = list(base)
        edits = 1 if rng.random() < 0.6 else 2
        for _ in range(edits):
            seg = segments[int(rng.integers(0, len(segments)))]
            if len(seg) <= 1:
                continue
            if rng.random() < 0.6:
                i, j = rng.choice(seg, size=2, replace=False)
                order[int(i)], order[int(j)] = order[int(j)], order[int(i)]
            else:
                span = min(max_shuffle_span, len(seg))
                if span < 2:
                    continue
                w = int(rng.integers(2, span + 1))
                start = int(rng.integers(0, len(seg) - w + 1))
                idx_slice = seg[start:start + w]
                vals = [order[i] for i in idx_slice]
                rng.shuffle(vals)
                for k, idx in enumerate(idx_slice):
                    order[idx] = vals[k]

        t = tuple(order)
        if t not in seen:
            seen.add(t)
            orders.append(t)

    return orders

def optimize_sign_order(signs: List[str], max_combinations: int = 20, min_improvement: float = 0.5):
    signs = list(signs)
    if len(signs) <= 2:
        return signs, {'used': False, 'reason': 'sequence too short', 'tested_orders': 1}

    _, base_score, _, _, _ = _decode_signs(signs, decode_beam_size=3)
    orders = _generate_local_orders(len(signs), max_combinations=max_combinations, n_segments=3, max_shuffle_span=3)

    best_order = orders[0]
    best_score = base_score
    for order in orders:
        cand = [signs[i] for i in order]
        _, sc, _, _, _ = _decode_signs(cand, decode_beam_size=3)
        if sc > best_score:
            best_score = sc
            best_order = order

    improved = (best_score - base_score) >= min_improvement
    best_signs = [signs[i] for i in best_order] if improved else signs
    info = {
        'used': improved,
        'reason': 'better local permutation found' if improved else 'no meaningful gain',
        'score_before': float(base_score),
        'score_after': float(best_score),
        'score_gain': float(best_score - base_score),
        'tested_orders': len(orders),
        'max_combinations': max_combinations
    }
    return best_signs, info

def translit_to_english(translit_sentence: str, max_new_tokens: int = 64):
    src = TASK_PREFIX + translit_sentence
    inputs = tokenizer(src, return_tensors='pt', truncation=True, max_length=MAX_SOURCE_LEN)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        gen = tr_model.generate(**inputs, max_new_tokens=max_new_tokens, num_beams=6, no_repeat_ngram_size=3, early_stopping=True)
    return tokenizer.decode(gen[0], skip_special_tokens=True).strip()

def keywords_to_sentence(sentence: str):
    prompt = (
        'Rewrite into natural English. Remove repeated phrases. Keep original meaning. ' +
        'If the sentence does not make sense, make it meaningful and grammatically correct. ' +
        'Do not add new facts. Sentence: ' + str(sentence).strip()
    )
    inputs = flan_tokenizer(prompt, return_tensors='pt', truncation=True, max_length=256)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = flan_model.generate(
            **inputs,
            num_beams=8,
            no_repeat_ngram_size=3,
            repetition_penalty=1.4,
            min_new_tokens=4,
            max_new_tokens=50,
            do_sample=False
        )
    return flan_tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

def gardiner_sequence_to_translit(gardiner_input, try_reorder: bool = True, max_combinations: int = 20, return_debug: bool = True):
    signs = parse_gardiner_input(gardiner_input)
    translit, score, changes, segments, exact_ratio = _decode_signs(signs, decode_beam_size=6)

    chosen_signs = signs
    reorder_info = {'used': False, 'reason': 'not attempted', 'tested_orders': 1}
    if try_reorder and 4 < len(signs) <= 48 and exact_ratio < 0.70:
        reordered, reorder_info = optimize_sign_order(signs, max_combinations=max_combinations, min_improvement=0.5)
        if reorder_info.get('used', False):
            chosen_signs = reordered
            translit, score, changes, segments, exact_ratio = _decode_signs(chosen_signs, decode_beam_size=6)

    dbg = {
        'normalized_signs': signs,
        'chosen_sign_order': chosen_signs,
        'auto_corrections': changes,
        'decode_score': score,
        'segments': segments,
        'exact_segment_ratio': exact_ratio,
        'reorder_info': reorder_info,
    }
    return (translit, dbg) if return_debug else translit

def gardiner_to_english(gardiner_input, try_reorder: bool = True, max_combinations: int = 20, return_debug: bool = False):
    translit, dbg = gardiner_sequence_to_translit(gardiner_input, try_reorder=try_reorder, max_combinations=max_combinations, return_debug=True)
    english = translit_to_english(translit, max_new_tokens=64)
    if return_debug:
        return translit, english, dbg
    return translit, english

def print_grouping_debug(debug_obj, max_rows: int = 30):
    print(f"Grouping quality: exact segment ratio = {debug_obj['exact_segment_ratio']:.2f}")
    print('Auto-corrections:', debug_obj['auto_corrections'])
    print('Reorder info:', debug_obj['reorder_info'])
    for i, seg in enumerate(debug_obj['segments'][:max_rows], 1):
        tag = 'EXACT' if seg['exact'] else f"FUZZY(d={seg['distance']:.2f})"
        left = ' '.join(seg['input_signs'])
        print(f"{i:02d}. [{tag}] {left} -> {seg['translit']}")

def model4_demo_from_split_line(split: str = 'train', line_index: int = 0, max_combinations: int = 20, show_debug: bool = True):
    split = split.lower().strip()
    if split not in SPLIT_DATA:
        raise ValueError("split must be one of: 'train', 'val', 'test'")

    egy_lines = SPLIT_DATA[split]['egy']
    eng_lines = SPLIT_DATA[split]['eng']
    if line_index < 0 or line_index >= len(egy_lines):
        raise IndexError(f'line_index out of range for split {split}. Valid range: 0..{len(egy_lines)-1}')

    input_line = egy_lines[line_index]
    ground_truth = eng_lines[line_index]
    translit, predicted, dbg = gardiner_to_english(
        input_line,
        try_reorder=True,
        max_combinations=max_combinations,
        return_debug=True
    )

    print(f'Split: {split} | line[{line_index}]')
    print('Input line (Egyptian transliteration):', input_line)
    print('Ground truth English:', ground_truth)
    print('Predicted English:', predicted)
    print('Resolved transliteration:', translit)

    if show_debug:
        print_grouping_debug(dbg)

    return {
        'split': split,
        'line_index': line_index,
        'input_line': input_line,
        'ground_truth': ground_truth,
        'predicted': predicted,
        'resolved_transliteration': translit,
        'debug': dbg,
    }

In [ ]:
# Model 4 split-line demo execution (input line + ground truth + predicted)
# You can change split to 'train', 'val', or 'test' and choose any line index.
model4_split_demo = model4_demo_from_split_line(
    split='train',
    line_index=0,
    max_combinations=20,
    show_debug=False
)

In [ ]:
# ------------------------------
# Model 4 and 5 demo from classifier output
# ------------------------------
def class_results_to_sequence(classification_results: List[Dict]):
    return [r['class_name'] for r in classification_results]

def demonstrate_model4_model5_from_classification(classification_results: List[Dict], max_combinations: int = 20, show_debug: bool = True):
    class_sequence = class_results_to_sequence(classification_results)
    translit, en_raw, dbg = gardiner_to_english(class_sequence, try_reorder=True, max_combinations=max_combinations, return_debug=True)
    en_fixed = keywords_to_sentence(en_raw)

    print('Classifier class sequence:', class_sequence)
    print('Resolved transliteration:', translit)
    print('Model 4 translation:', en_raw)
    print('Model 5 grammar fixed:', en_fixed)
    if show_debug:
        print_grouping_debug(dbg)

    return {
        'class_sequence': class_sequence,
        'transliteration': translit,
        'model4_translation': en_raw,
        'model5_grammar_fixed': en_fixed,
        'debug': dbg
    }

model45_result = demonstrate_model4_model5_from_classification(folder_class_demo, max_combinations=20, show_debug=True)

In [ ]:
# ------------------------------
# Full end-to-end demo runner (Model 1 -> 5)
# ------------------------------
def run_full_pipeline(
    image_path: str,
    use_model1_to_get_line: bool = False,
    line_crop_index: int = 0,
    max_combinations: int = 20,
    apply_grammar_fix: bool = True,
    show_debug: bool = True,
    show_classified_crops: bool = True
):
    if not os.path.exists(image_path):
        raise FileNotFoundError(f'Input image not found: {image_path}')

    plt.figure(figsize=(10, 6))
    plt.imshow(Image.open(image_path).convert('RGB'))
    plt.title('Input Image')
    plt.axis('off')
    plt.show()

    target_line_image = image_path
    line_crops = []
    if use_model1_to_get_line:
        print('Stage 1: Model 1 line/cartouche detection and cropping')
        line_crops = save_line_crops_from_wall_image(
            wall_image_path=image_path,
            model1=model1,
            out_dir=MODEL1_CROP_OUTPUT,
            conf=0.25
        )
        if not line_crops:
            raise RuntimeError('Model 1 did not return any line crops for this image.')
        if line_crop_index < 0 or line_crop_index >= len(line_crops):
            raise IndexError(f'line_crop_index out of range. Valid range: 0..{len(line_crops)-1}')
        target_line_image = line_crops[line_crop_index]
        print('Chosen line crop from Model 1:', target_line_image)

    print('Stage 2: Model 2 glyph detection and crop enhancement')
    model2_out = detect_glyphs_and_save_enhanced(
        line_image_path=target_line_image,
        model2=model2,
        out_dir=MODEL2_GLYPH_CROP_OUTPUT,
        conf=0.4,
        margin=4,
        scale_factor=3,
        visualize=True
    )

    print('Stage 3: Model 3 glyph classification')
    class_results = classify_folder(MODEL2_GLYPH_CROP_OUTPUT)
    if show_classified_crops:
        show_classified_gray_crops(class_results, max_items=24, cols=4)
    class_sequence = class_results_to_sequence(class_results)

    print('Stage 4: Model 4 translation')
    translit, en_raw, dbg = gardiner_to_english(
        class_sequence,
        try_reorder=True,
        max_combinations=max_combinations,
        return_debug=True
    )

    print('Recognized Gardiner classes:', class_sequence)
    print('Resolved transliteration:', translit)
    print('Model 4 translation:', en_raw)

    en_final = en_raw
    if apply_grammar_fix:
        print('Stage 5: Model 5 grammar correction')
        en_final = keywords_to_sentence(en_raw)
        print('Model 5 grammar fixed:', en_final)

    if show_debug:
        print_grouping_debug(dbg)

    return {
        'line_crops_from_model1': line_crops,
        'line_image_used': target_line_image,
        'model2_boxes': model2_out['boxes'],
        'class_sequence': class_sequence,
        'transliteration': translit,
        'model4_translation': en_raw,
        'model5_translation': en_final,
        'debug': dbg
    }

def run_model1_to_model4_flow(wall_image_path: str, line_crop_index: int = 0, max_combinations: int = 20, show_debug: bool = True):
    return run_full_pipeline(
        image_path=wall_image_path,
        use_model1_to_get_line=True,
        line_crop_index=line_crop_index,
        max_combinations=max_combinations,
        apply_grammar_fix=False,
        show_debug=show_debug,
        show_classified_crops=True
    )

In [ ]:
# Final demonstrations
# Option A: fast path (line image directly to Model 2 -> 3 -> 4 -> 5)
full_demo_result = run_full_pipeline(
    image_path=line_image_for_model2,
    use_model1_to_get_line=False,
    max_combinations=20,
    apply_grammar_fix=True,
    show_debug=True,
    show_classified_crops=True
)

# Option B: explicit flow path (Model 1 output crop -> Model 2 -> 3 -> 4)
# Use wall image, choose line_crop_index to select which Model 1 crop enters the downstream flow.
model1_to_model4_flow_result = run_model1_to_model4_flow(
    wall_image_path=DEMO_WALL_IMAGE_PATH,
    line_crop_index=0,
    max_combinations=20,
    show_debug=True
)